## Imports

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from setup_lookup_table import setup_lookup_table
from create_event_sequence import create_event_sequence
from flood_adapt.objects.forcing.unit_system import UnitTypesLength
from flood_adapt import FloodAdapt
from flood_adapt.config.config import Settings


## Inputs 
Provide the event set and the range of SLR values to be calculated. 

In [ ]:
# FloodAdapt databse inputs
DATA_DIR = Path(r"c:\Users\winter_ga\Offline_data\FloodAdapt-WorkingDatabase\Charleston\4_FloodAdapt\Database")
site = "charleston_beta_release"
name_event_set = "event_set_coastal"
SFINCS_BIN_PATH=Path(r"c:\Users\winter_ga\Offline_data\FloodAdapt-Database\system\win-64\sfincs_v.2.1_alpha\sfincs.exe")
FIAT_BIN_PATH=Path(r"c:\Users\winter_ga\Offline_data\FloodAdapt-Database\system\win-64\fiat_v0.2.1\fiat.exe")

# scenario input parameters
slr = np.arange(0, 3.1, 1)
unit = UnitTypesLength.feet
fp_height = 2  # floodproof height 

# Monte Carlo input parameters for event sequences
timestep = 1  # timestep of Monte Carlo in years
years = 30  # length of simulation in years
n_seq = 20  # number of event sequences to generate
seed = 42  # random seed for reproducibility

# SLR scenario to evaluate damages over time
slr_scenario_name = "NOAA High" # This needs to be a scenario in the FloodAdapt database

## Initialize FloodAdapt Database

In [ ]:
    # Initialize FloodAdapt
settings = Settings(
    DATABASE_ROOT=DATA_DIR,
    DATABASE_NAME=site,
    SFINCS_BIN_PATH=SFINCS_BIN_PATH,
    FIAT_BIN_PATH=FIAT_BIN_PATH,
    VALIDATE_BINARIES=True,
)
fa = FloodAdapt(database_path=settings.database_path)

## Create event sequence and look up table

To Do: 
- naming of projections and strategies needs to be more unique to avoid confusion when running a different analysis.

In [ ]:
# Lookup table

ds_impacts = setup_lookup_table(
        fa, 
        name_event_set, 
        slr=np.arange(0, 3.1, 1), 
        unit = unit, 
        fp_height = 2, 
        timestep=1, 
        run_scenarios=False
        )


In [ ]:

# Event sequence
fn_event_set = DATA_DIR / site / "input" / "events" / name_event_set / f"{name_event_set}.toml"

[occ, sequences, event_names, probs] = create_event_sequence(
        fn_event_set,   
        years=years, 
        n_seq=n_seq, 
        dt=timestep,
        seed=seed)

## Plot event sequences

In [ ]:

# Create figure with 1 row, 3 columns
fig, axes = plt.subplots(1,1, figsize=(8,8))
seq_max = 20

# Color map for events
colors = {event_id: plt.cm.tab10(i) for i, event_id in enumerate(event_names)}
yticklabels = []
# Plot first seq_max sequences
for seq_idx in range(min(seq_max, len(sequences))):
    ax = axes#[seq_idx]
    
    # For each year, plot events stacked vertically
    for year_idx, year_events in enumerate(sequences[seq_idx]):
        for marker_offset, event_id in enumerate(sorted(year_events)):
            ax.scatter(year_idx, marker_offset*0.15+seq_idx, 
                      color=colors[event_id], 
                      s=50, 
                      marker='o',
                      label=event_id if year_idx == 0 else "")
    yticklabels.append(f"Seq {seq_idx}")
    
ax.set_xlabel('Year')
ax.set_yticks([i for i in range(min(seq_max, len(sequences)))])
ax.set_yticklabels(yticklabels)
# ax.set_title(f'Sequence {seq_idx}')
ax.set_xlim(-0.5, years - 0.5)
# ax.set_ylim(-0.1, 1)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, years, 5))

# Add legend (use unique events)
handles = [plt.scatter([], [], color=colors[e], s=200, marker='o') for e in event_names]
fig.legend(handles, event_names, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=len(event_names))

plt.tight_layout()
plt.show()

fig.savefig('event_sequences.png', bbox_inches='tight')

## Step 6: Plot impacts for event sequences

To Do: 
- Improve SLR projection: How - Ceil? Flood? Linear?

In [ ]:
seq_idx = 13  # just evaluate for first sequence for now
year_eval = 2020 + np.arange(years)
slr_values = []
damage=np.zeros((len(ds_impacts.strategy.values), len(sequences), years, len(event_names)))

width = 0.25  # the width of the bars
fig, ax = plt.subplots(figsize=(8,6))

clrs = {event_name: plt.cm.tab10(i) for i, event_name in enumerate(event_names)}

# for seq_idx in range(len(sequences)):
for str_idx, strategy in enumerate(ds_impacts.strategy.values):
    multiplier = 0 # used to offset bars
    for ii, year_events in enumerate(sequences[seq_idx]):
        slr_values.append(fa.interp_slr(slr_scenario=slr_scenario_name, year=year_eval[ii]))
        for nn,event in enumerate(event_names):
            if event in year_events:    
                damage_high = ds_impacts["total_damage"].sel(slr=np.ceil(slr_values[ii]),strategy=ds_impacts.strategy.values[str_idx],event=event).sum()      
                damage_low = ds_impacts["total_damage"].sel(slr=np.floor(slr_values[ii]),strategy=ds_impacts.strategy.values[str_idx],event=event).sum()      
                damage[str_idx,seq_idx,ii,nn] = np.interp(slr_values[ii], 
                                    [np.floor(slr_values[ii]), np.ceil(slr_values[ii])], 
                                    [damage_low, damage_high])               

    for event_name, dam in zip(event_names, damage[str_idx,seq_idx,:,:].T):
        offset = width * multiplier
        rects = ax.bar(np.array(year_eval) + offset, dam, width, label=f"{event_name} - {ds_impacts.strategy.values[str_idx]}",

                          color=clrs[event_name],alpha=0.5+0.5*str_idx)
        # ax.bar_label(rects, padding=3)
        multiplier += 1

ax.set_ylabel("Total Damage ($)")
ax.set_title(f"Annual Damage by Event Type")
ax.legend(loc='upper left', ncols=1)
        

## Update adaptation every time step